In [1]:
import os

In [2]:
import random

In [3]:
import numpy as np

In [4]:
import pandas as pd

In [5]:
import matplotlib.pyplot as plt

In [6]:
import seaborn as sns

In [7]:
import warnings


In [8]:
warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# -----------------------------------------------------------------------------
sns.set_theme(style='whitegrid', palette='colorblind', font='DejaVu Sans')
CHART_DPI = 150  # Resolución para guardar los gráficos

In [9]:
def cargar_datos(path_train='solicitudes_train.csv', 
                 path_test='solicitudes_test.csv', 
                 path_bancario='comportamiento_bancario.csv'):
    """
    Carga los tres archivos CSV de la competencia.
    """
    print("--- 1. Carga de Datos ---")
    train = pd.read_csv(path_train) if os.path.exists(path_train) else None
    test = pd.read_csv(path_test) if os.path.exists(path_test) else None
    bancario = pd.read_csv(path_bancario) if os.path.exists(path_bancario) else None
    
    if train is not None:
        print(f"train shape: {train.shape}")
    if test is not None:
        print(f"test shape: {test.shape}")
    if bancario is not None:
        print(f"bancario shape: {bancario.shape}")
        
    return train, test, bancario

In [10]:
df_train,df_test,df_bancario=cargar_datos()

--- 1. Carga de Datos ---
train shape: (32416, 8)
test shape: (3061, 7)
bancario shape: (661139, 4)


In [11]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32416 entries, 0 to 32415
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_Solicitud      32416 non-null  int64  
 1   ID_Usuario        32416 non-null  int64  
 2   Monto_Solicitado  32416 non-null  int64  
 3   Tasa_Interes      32416 non-null  float64
 4   Ingreso_Mensual   32416 non-null  int64  
 5   Motivo_Prestamo   32416 non-null  object 
 6   Fecha_Solicitud   32416 non-null  object 
 7   Estado_Final      32416 non-null  int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 2.0+ MB


In [12]:
df_bancario.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 661139 entries, 0 to 661138
Data columns (total 4 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID_Usuario                661139 non-null  int64  
 1   Mes_Referencia            661139 non-null  object 
 2   Saldo_Promedio_Mensual    661139 non-null  float64
 3   Dias_Atraso_Otras_Deudas  661139 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 20.2+ MB


In [13]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3061 entries, 0 to 3060
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_Solicitud      3061 non-null   int64  
 1   ID_Usuario        3061 non-null   int64  
 2   Monto_Solicitado  3061 non-null   int64  
 3   Tasa_Interes      3061 non-null   float64
 4   Ingreso_Mensual   3061 non-null   int64  
 5   Motivo_Prestamo   3061 non-null   object 
 6   Fecha_Solicitud   3061 non-null   object 
dtypes: float64(1), int64(4), object(2)
memory usage: 167.5+ KB


In [14]:
def eda_solicitudes(train, output_dir='graficos_eda'):
    """
    Análisis Exploratorio para solicitudes_train.csv
    """
    os.makedirs(output_dir, exist_ok=True)
    print("\n--- 2. Análisis Exploratorio de Solicitudes (Train) ---")
    
    # 2.1 Información General y Nulos
    print("\n> Resumen de Valores Faltantes:")
    nulos = train.isnull().sum()
    if nulos.sum() > 0:
        print(nulos[nulos > 0])
    else:
        print("  ✓ No se encontraron valores faltantes en el conjunto de entrenamiento.")
    
    # 2.2 Balance de Variable Objetivo
    if 'Estado_Final' in train.columns:
        prop = train['Estado_Final'].value_counts(normalize=True) * 100
        print(f"\n> Balance de Clase ('Estado_Final'):")
        print(f"  - Pagados  (1): {prop.get(1, 0):.2f}%")
        print(f"  - Defaults (0): {prop.get(0, 0):.2f}%")
        
        # Gráfico 1: Target Balance
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.countplot(data=train, x='Estado_Final', ax=ax)
        ax.set_title('Distribución de Estado_Final (0: Default, 1: Pagado)', fontsize=12, fontweight='bold')
        
        # Corrección explícita de ticks y labels
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Default (0)', 'Pagado (1)'])
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '01_variable_objetivo.png'), dpi=CHART_DPI)
        plt.close()

    # 2.3 Distribución de Variables Financieras Clave
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Distribución de Variables Financieras Clave en Solicitudes', fontsize=14, fontweight='bold')
    
    if 'Monto_Solicitado' in train.columns:
        sns.histplot(data=train, x='Monto_Solicitado', hue='Estado_Final', kde=True, ax=axes[0], element='step')
        axes[0].set_title('Monto Solicitado')
        
    if 'Tasa_Interes' in train.columns:
        sns.histplot(data=train, x='Tasa_Interes', hue='Estado_Final', kde=True, ax=axes[1], element='step')
        axes[1].set_title('Tasa de Interés')
        
    if 'Ingreso_Mensual' in train.columns:
        sns.histplot(data=train, x='Ingreso_Mensual', hue='Estado_Final', kde=True, ax=axes[2], element='step')
        axes[2].set_title('Ingreso Mensual')
        
    sns.despine()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '02_financieras_dist.png'), dpi=CHART_DPI)
    plt.close()
    print(f"  ✓ Gráficos guardados exitosamente en la carpeta '{output_dir}/'.")

In [15]:
eda_solicitudes(df_train)


--- 2. Análisis Exploratorio de Solicitudes (Train) ---

> Resumen de Valores Faltantes:
  ✓ No se encontraron valores faltantes en el conjunto de entrenamiento.

> Balance de Clase ('Estado_Final'):
  - Pagados  (1): 84.05%
  - Defaults (0): 15.95%
  ✓ Gráficos guardados exitosamente en la carpeta 'graficos_eda/'.


In [16]:
def eda_comportamiento_bancario(bancario, output_dir='graficos_eda'):
    """
    Análisis Exploratorio para comportamiento_bancario.csv
    """
    os.makedirs(output_dir, exist_ok=True)
    print("\n--- 3. Análisis Exploratorio de Comportamiento Bancario ---")
    
    # 3.1 Registros por Usuario
    meses_por_user = bancario.groupby('ID_Usuario')['Mes_Referencia'].count()
    print(f"\n> Registros temporales por usuario:")
    print(f"  - Mínimo de meses: {meses_por_user.min()}")
    print(f"  - Promedio de meses: {meses_por_user.mean():.1f}")
    print(f"  - Máximo de meses: {meses_por_user.max()}")
    
    # 3.2 Días de Atraso en Deudas
    if 'Dias_Atraso_Otras_Deudas' in bancario.columns:
        morosos = (bancario['Dias_Atraso_Otras_Deudas'] > 0).sum()
        pct_morosos = (morosos / len(bancario)) * 100
        print(f"\n> Registros mensuales con atraso (> 0 días): {morosos} de {len(bancario)} ({pct_morosos:.2f}%)")
        
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(bancario['Dias_Atraso_Otras_Deudas'], bins=30, kde=True, ax=ax)
        ax.set_title('Distribución de Días de Atraso en Otras Deudas', fontsize=12, fontweight='bold')
        ax.set_xlabel('Días de Atraso')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '03_atrasos_dist.png'), dpi=CHART_DPI)
        plt.close()
        
    # 3.3 Evaluación del Saldo Promedio
    if 'Saldo_Promedio_Mensual' in bancario.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(x=bancario['Saldo_Promedio_Mensual'], ax=ax)
        ax.set_title('Detección de Outliers en Saldo Promedio Mensual', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, '04_saldo_promedio.png'), dpi=CHART_DPI)
        plt.close()
    
    print(f"  ✓ Gráficos bancarios guardados exitosamente en la carpeta '{output_dir}/'.")

In [21]:
eda_comportamiento_bancario(df_bancario)


--- 3. Análisis Exploratorio de Comportamiento Bancario ---

> Registros temporales por usuario:
  - Mínimo de meses: 17
  - Promedio de meses: 22.8
  - Máximo de meses: 24

> Registros mensuales con atraso (> 0 días): 522614 de 661139 (79.05%)
  ✓ Gráficos bancarios guardados exitosamente en la carpeta 'graficos_eda/'.
